# Specific Stream Power QA
A quick look at the results of the specific stream power estimation.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

DATA_DIR = Path("data/")

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')


# CONUS extent
CONUS_EXTENT = [-125, -66, 24, 50]  # [lon_min, lon_max, lat_min, lat_max]

def make_conus_ax(fig, pos=111, title=''):
    """Return a cartopy GeoAxes set to CONUS with standard features."""
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,       facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,      facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,      facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,     linewidth=0.4,       zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE,  linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,    linewidth=0.6,       zorder=3)
    if title:
        ax.set_title(title)
    return ax

In [ ]:
site_info = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")
coverage = pd.read_parquet(DATA_DIR / "metadata" / "data_coverage.parquet")
power = pd.read_parquet(DATA_DIR / "metadata" / "stream_power.parquet")

## 1. Distributions

In [ ]:
plot_params = ['action_ssp_wm2', 'flood_ssp_wm2', 'moderate_ssp_wm2', 'major_ssp_wm2']
# recession_params = ['ALPHA', 'BETA', 'Error']
# recession_params = ['Rs', 'Rb1', 'Rb2']

fig, axes = plt.subplots(1, len(plot_params), figsize=(16, 3), dpi=300)
for ax, col in zip(axes, plot_params):
    data = power[col].dropna()
    ax.hist(data, bins=50, edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.2,
               label=f'median={data.median():.3g}')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.legend(fontsize=8)
    

plt.tight_layout()
plt.show()

## 2. Maps of Specific Stream Power

In [ ]:
map_df = power.merge(
    site_info[["site_no", "latitude", "longitude", "station_name"]],
    on="site_no", how="left"
).dropna(subset=["latitude", "longitude"])

In [ ]:
map_params = ['action_ssp_wm2', 'flood_ssp_wm2', 'moderate_ssp_wm2', 'major_ssp_wm2']

fig = plt.figure(figsize=(18, 12), dpi=300)
for i, col in enumerate(map_params, 1):
    sub  = map_df.dropna(subset=['latitude', 'longitude', col])
    # norm = mcolors.Normalize(vmin=sub[col].quantile(0.02), vmax=sub[col].quantile(0.98))
    norm = mcolors.LogNorm(vmin=sub[col].quantile(0.02), vmax=sub[col].quantile(0.98))
    ax   = make_conus_ax(fig, pos=(2, 2, i), title=col)
    sc   = ax.scatter(
        sub['longitude'], sub['latitude'],
        c=sub[col], cmap='plasma', norm=norm,
        s=14, alpha=0.8, linewidths=0,
        transform=ccrs.PlateCarree(), zorder=4
    )
    plt.colorbar(sc, ax=ax, orientation='horizontal', shrink=0.8, pad=0.04, label=col)

fig.suptitle('Spatial distribution of Specific Stream Power', fontsize=13)
plt.tight_layout()
plt.show()